In [1]:
import joblib
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)
from sklearn.model_selection import train_test_split


df = pd.read_csv("../data/processed/model_data.csv")
X = df.drop("default", axis=1)
y = df["default"]

_, X_test, _, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

logistic_model = joblib.load("../models/logistic_regression.pkl")
random_forest_model = joblib.load("../models/random_forest.pkl")
xgb_model = joblib.load("../models/xgboost.pkl")

In [2]:
models = {
    "Logistic Regression": logistic_model,
    "Random Forest": random_forest_model,
    "XGBoost": xgb_model
}

results = []

for name, model in models.items():

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred),
        "ROC AUC": roc_auc_score(y_test, y_prob)
    })

results_df = pd.DataFrame(results)

results_df

,Model,Accuracy,Precision,Recall,F1 Score,ROC AUC
0,Logistic Regression,0.687336,0.385569,0.723554,0.503064,0.779222
1,Random Forest,0.890483,0.901361,0.560649,0.691304,0.924443
2,XGBoost,0.870122,0.809677,0.531030,0.641397,0.866090


In [3]:
best_model_name = results_df.loc[results_df['ROC AUC'].idxmax()]['Model']
best_model = models[best_model_name]

joblib.dump(
    best_model,
    "../models/best_model.pkl"
)

['../models/best_model.pkl']